# E1+E2 — Per-ε hyperparameter optimization + final audited models (v1, 2026-07-11)

Implements plan §E1/§E2 (`research/EXPERIMENT_PLAN_2026-07-08.md`). Run this notebook **before** the
E4 decomposition sweep — E4 audits ONLY the tuned models produced here.

**What it does, per dataset:**
1. **E1** — for each target ε ∈ {0.3, 0.5, 1.0, 2.0, 4.0, 8.0}: 30 Optuna TPE trials over
   (lr, batch, epochs, clip); σ re-solved per trial so eps_PLD == target exactly (google backend, hard — I2).
   Trial seeds live in the 900000+/910000+ band and never touch canonical seeds (I6).
2. **E2** — trains the winning config with the canonical seed triple 123/124/125 and saves checkpoints.

**Session plan:** one dataset per session; ~20–60 min/ε-cell on T4 for E1 (6 cells) + E2 trainings.
`EPSILONS` can be trimmed to split across sessions — every cell is independent and the notebook
zips partial results after each cell, so a disconnect loses at most one cell.

**Acceptance gates baked in:** achieved ε within 1e-3 of target (E1); 3 seeds/cell, accuracy std < 2%,
recorded eps_PLD == target ± 1e-3 with the google backend (E2/G1).

Upload `hpo_bundle_v1.zip` when prompted.

Bundle sha256: `44cfbb9e9190d392036f28ae03e4639e37bc36f02e8ac9c64959faf51df935fe`

In [ ]:
from google.colab import files
up = files.upload()  # choose hpo_bundle_v1.zip
import zipfile, hashlib
zname = [f for f in up if f.endswith(".zip")][0]
zipfile.ZipFile(zname).extractall(".")
print("Extracted", zname)
digest = hashlib.sha256(open(zname, "rb").read()).hexdigest()
expected = "44cfbb9e9190d392036f28ae03e4639e37bc36f02e8ac9c64959faf51df935fe"
assert digest == expected, f"BUNDLE MISMATCH: {digest} != {expected} -- wrong or corrupted zip"
print("sha256 verified:", digest)

In [ ]:
%pip -q install opacus dp-accounting pyyaml scipy optuna ucimlrepo
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY -- ABORT: runtime will be ~10-20x slower; Runtime > Change runtime type > T4 GPU")

In [ ]:
# --- PRE-FLIGHT: bundled test suites + I2/I6 markers (aborts) ---------------
import subprocess, sys
for suite in ("tests.test_sigma_solver", "tests.test_empirical"):
    r = subprocess.run([sys.executable, "-m", "unittest", suite, "-v"],
                       capture_output=True, text=True)
    print(r.stderr[-2000:])
    assert r.returncode == 0, f"TEST SUITE FAILED ({suite}) -- do NOT run E1."
sys.path.insert(0, "src")
from dp_audit_tightness.privacy.pld_accounting import compute_epsilon_pld
res = compute_epsilon_pld(noise_multiplier=1.1, sampling_rate=128/2048, num_steps=16, delta=1e-5)
assert res["backend_used"] == "google_dp_accounting", f"I2 violation: {res}"
tuner_src = open("experiments/tune_hyperparams.py").read()
assert "base_trial_training_seed + trial.number" in tuner_src, "I6 marker missing (trial seed band)"
assert "restore canonical seeds" in tuner_src, "I6 marker missing (canonical restore)"
print("\nE1/E2 pre-flight OK -- safe to run.")

In [ ]:
# --- PARAMETERS --------------------------------------------------------------
DATASET = "mnist"                              # "mnist" | "diabetes" -- one dataset per session
EPSILONS = [0.3, 0.5, 1.0, 2.0, 4.0, 8.0]      # trim to split E1 across sessions
CONFIG = f"configs/hpo/{DATASET}_eps_grid.yaml"
print(DATASET, EPSILONS, CONFIG)

In [ ]:
# --- Data prep ---------------------------------------------------------------
import subprocess, sys
from pathlib import Path
if DATASET == "diabetes":
    if not Path("data/raw/cdc_diabetes.npz").exists():
        r = subprocess.run([sys.executable, "experiments/prepare_diabetes.py", "--fetch-uci"])
        assert r.returncode == 0, "cdc_diabetes.npz build failed"
    print("cdc_diabetes.npz ready")
else:
    print("MNIST downloads automatically on first use (torchvision).")

In [ ]:
# --- E1: one Optuna study per target epsilon (resumable per cell) ------------
import subprocess, sys, time, shutil
for eps in EPSILONS:
    started = time.time()
    print(f"=== E1 {DATASET} eps={eps} (30 trials) ===", flush=True)
    r = subprocess.run([sys.executable, "experiments/tune_hyperparams.py",
                        "--config", CONFIG, "--epsilon", str(eps)])
    assert r.returncode == 0, f"HPO cell eps={eps} failed"
    print(f"cell eps={eps} done in {(time.time()-started)/60:.1f} min")
    # checkpoint after every cell: a disconnect loses at most one cell
    shutil.make_archive(f"hpo_partial_{DATASET}", "zip", ".", "results/hpo")
print("E1 complete for", DATASET)

In [ ]:
# --- E1 acceptance (plan SE1): achieved eps within 1e-3; YAML loadable -------
import json, sys
from pathlib import Path
sys.path.insert(0, "src")
from dp_audit_tightness.config import load_train_config
ok = True
studies = sorted(Path("results/hpo").glob(f"hpo_{DATASET}_eps*.json"))
assert studies, "no study records found"
for path in studies:
    study = json.loads(path.read_text())
    target = study["target_epsilon"]
    achieved = study["best_user_attrs"]["achieved_epsilon"]
    load_train_config(Path("results/hpo/best_configs") / (path.stem + ".yaml"))
    good = abs(achieved - target) <= 1e-3
    ok &= good
    print(f"{path.stem}: best_acc={study['best_value_eval_accuracy']:.4f} "
          f"achieved_eps={achieved:.6f} target={target} {'OK' if good else 'FAIL'} "
          f"params={study['best_params']}")
assert ok, "E1 acceptance FAILED: achieved epsilon drifts >1e-3 from target"
print("E1 acceptance PASSED")

In [ ]:
# --- E2: final audited models, canonical seed triple (I6) ---------------------
import subprocess, sys
from pathlib import Path
best = sorted(Path("results/hpo/best_configs").glob(f"hpo_{DATASET}_eps*.yaml"))
assert best, "no best configs -- run E1 first"
for cfg in best:
    print(f"=== E2 {cfg.name} (seeds 123 124 125) ===", flush=True)
    r = subprocess.run([sys.executable, "experiments/run_train.py", "--config", str(cfg),
                        "--training-seeds", "123", "124", "125"])
    assert r.returncode == 0, f"E2 training failed for {cfg.name}"
print("E2 complete")

In [ ]:
# --- E2 acceptance / gate G1 --------------------------------------------------
import json, statistics
from pathlib import Path
groups = {}
for path in Path("results/training").glob("*.json"):
    rec = json.loads(path.read_text())
    if "_tuned_eps" not in rec.get("experiment_name", "") or rec.get("dataset") != DATASET:
        continue
    groups.setdefault(rec["experiment_name"], []).append(rec)
assert groups, "no tuned training runs found"
all_ok = True
for name, recs in sorted(groups.items()):
    target = float(name.rsplit("_tuned_eps", 1)[1])
    accs = [r["utility_metrics"]["accuracy"] for r in recs]
    std = statistics.pstdev(accs)
    eps_ok = all(r["epsilon_upper_pld"] is not None
                 and abs(r["epsilon_upper_pld"] - target) <= 1e-3 for r in recs)
    backend_ok = all(r["pld_accounting_backend"] == "google_dp_accounting" for r in recs)
    ok = len(recs) == 3 and std < 0.02 and eps_ok and backend_ok
    all_ok &= ok
    print(f"{name}: n={len(recs)} acc_mean={statistics.fmean(accs):.4f} acc_std={std:.4f} "
          f"eps_ok={eps_ok} backend_ok={backend_ok} -> {'OK' if ok else 'FAIL'}")
assert all_ok, "E2 acceptance FAILED (see rows above)"
print("E2 acceptance PASSED (G1)")

In [ ]:
# --- Download E1+E2 results (feed this zip to the E4 notebook) ----------------
import shutil
from google.colab import files
name = f"hpo_e1e2_results_{DATASET}"
shutil.make_archive(name, "zip", ".", "results")
files.download(name + ".zip")